# `SolverProtocols` — Architecture Overview

PyOMES has two orthogonal solver axes, each a thin, deliberately
minimal `typing.Protocol`:

- **Axis 1 — `StepSolver`.** Per-CV physics: one method,
  `solve_step(cv, dt_h, t_h, external_source_terms) -> AdvanceResult`.
  Advances a single `ControlVolume` by one macro timestep.
- **Axis 2 — `SystemSolver`.** Whole-system orchestration: one method,
  `advance_system(sim, dt_h, t_h)`. Given a whole `Simulation`
  (multiple CVs, inter-CV links, controllers, profiles), decides how
  one macro step gets composed.

They compose orthogonally — `Simulation(solver=..., system_solver=...)`
— pick any `StepSolver` per CV under any `SystemSolver`. Neither axis
has a privileged implementation: `solver=None` and `system_solver=None`
are pure sugar for one particular rung each (`SequentialAdvanceSolver`,
`ExplicitEulerSystemSolver`), not special-cased code paths.

This notebook is the map: what each protocol contract actually is, how
the shipped rungs on each axis compare, and which one to reach for.

## Notebooks in this folder

| Notebook | Covers |
|---|---|
| [01_writing_a_custom_solver.ipynb](01_writing_a_custom_solver.ipynb) | Writing your own `StepSolver` and `SystemSolver` — two worked examples, each inverting a real shipped solver's documented design decision |

Launch from the repo root with
`jupyter lab demos/features/SolverProtocols/`.

## Axis 1 — `StepSolver`: the three shipped rungs

| Solver | Semantics | Earns its keep when |
|---|---|---|
| `SequentialAdvanceSolver` (`solver=None` default) | Sequential operator split: speciation → feed → react → transfer; each sub-step sees post-previous state | Default path; small `dt_h`; simple models without strong feed/reaction coupling |
| `SimultaneousEulerSolver` | Simultaneous explicit Euler: all sub-systems read the same frozen snapshot, deltas summed and applied together, with swappable clamping | Larger `dt_h` where operator-splitting bias matters; matches the pre-refactor semantics ADM1 was validated against |
| `SimultaneousAdaptiveSolver` | `scipy.integrate.solve_ivp` adaptive; explicit (DOP853) or implicit (Radau, BDF) methods; optional speciation freeze (DAE-style, BSM2/PyADM1 convention) | Stiff models (BSM2 anaerobic digestion, fast H₂ kinetics); long macro timesteps explicit Euler cannot survive |

All three require only a `"liquid"` phase (speciation and reactions
are liquid-scoped) — a gas phase, solid phase, additional phases, or
none of those beyond liquid are all supported.

**Ownership guard.** If a CV is owned by a `Simulation`, calling
`cv.advance(...)` directly (rather than through `sim.run(...)`) emits
`OrchestrationWarning` — inter-CV links, controllers, and profiles
won't be applied for that step. Every `SystemSolver` calls the trusted
internal `cv._advance_unchecked(...)` instead, which bypasses the
guard.

### Instantiating each `StepSolver`

All three are plain classes — construct one and pass it as
`cv.advance(dt_h, t_h, solver=...)` or
`Simulation(solver=...)`/`Simulation(solver={cv_key: ...})`. Every
keyword below has a default, so `SolverClass()` with no arguments is
always valid; the "example" column shows a non-default configuration
for context.

| Solver | Default instantiation | Example with keywords | Keyword arguments |
|---|---|---|---|
| `SequentialAdvanceSolver` | `SequentialAdvanceSolver()` | `SequentialAdvanceSolver(clamp_fn=None)` | `clamp_fn` (default `proportional_clamp`) — swappable non-negativity clamp; `floor_clamp` or a bespoke composite are drop-in alternatives; `None` disables clamping entirely |
| `SimultaneousEulerSolver` | `SimultaneousEulerSolver()` | `SimultaneousEulerSolver(clamp_fn=floor_clamp)` | `clamp_fn` (default `proportional_clamp`) — same contract as above, applied once to the combined multi-source delta dict |
| `SimultaneousAdaptiveSolver` | `SimultaneousAdaptiveSolver()` | `SimultaneousAdaptiveSolver(method="Radau", rtol=1e-6, atol=1e-9)` | `method` (default `"DOP853"`) — `scipy.integrate.solve_ivp` method, `"Radau"`/`"BDF"` for stiff systems; `rtol`/`atol` (default `1e-6`/`1e-9`) — solver tolerances; `max_step` (default `1.0` h) — caps the internal adaptive step; `freeze_speciation` (default `False`) — solve speciation once per macro step instead of at every derivative evaluation (BSM2/PyADM1 DAE convention); `use_engine_jacobian` (default `False`) — use the speciation engine's analytical `jacobian_dz_dy()` instead of finite differences (only with a `GrayBoxEngineProtocol` engine and an implicit `method`) |

`clamp_fn`/`floor_clamp`/`proportional_clamp` live in
[`src/core/clamping.py`](../../../src/core/clamping.py) — see the
Clamping section below.

## Clamping — non-negativity handling

All three `StepSolver`s restore a physical invariant (non-negative
inventory) the raw numerics don't guarantee on their own:

| Function | Used by | Behaviour |
|---|---|---|
| `proportional_clamp` (default) | `SequentialAdvanceSolver`, `SimultaneousEulerSolver` | Scales a removal rate down so the result lands at exactly zero, preserving relative stoichiometry |
| `floor_clamp` | Either discrete-step solver, passed explicitly | Per-species floor (optionally at `eps`) — simpler, doesn't preserve relative stoichiometry |
| `floor_nonnegative` | `SimultaneousAdaptiveSolver` (always, not swappable) | Floors a raw ODE state array — a different failure mode (adaptive-integrator overshoot), not a `(deltas, current_mol, dt_h)` decision |

`clamp_fn` is a plain callable operating on an **arbitrary subset** of
species — this is what lets a bespoke composite match an external
reference model's specific per-species convention, using the shared
functions as ingredients rather than an all-or-nothing choice.
`clamp_fn=None` disables clamping entirely (a genuine way to let a
species go negative for diagnosis — `AccuracyMonitor.check_negative_mole`
flags it); `AccuracyMonitor.check_clamp_invoked` independently flags
when clamping actually changed a rate, naming the before/after values.

## Axis 2 — `SystemSolver`: the five shipped rungs

| Solver | Sequence per macro step | Earns its keep when |
|---|---|---|
| `ExplicitEulerSystemSolver` (default) | `profiles(t) → links(dt) → cv.advance(dt) → controllers` | τ_CFL ≫ dt_h and slow controllers — the common case |
| `StrangSplittingSystemSolver` | `profiles(t) → links(dt/2) → cv.advance(dt) → links(dt/2) → controllers` | Free 2nd-order splitting upgrade over Euler, ~zero extra cost |
| `MultirateSystemSolver` | `profiles(t) → cv.advance(dt) → [links(dt/M)] × M → controllers` | Fast inter-CV circulation (τ_CFL comparable to dt_h) without a global step-size cut |
| `ImplicitTransportSystemSolver` | Sparse implicit link-flux solve, unconditionally stable | Many-CV compartmental transport models (HPLC, packed beds), no tight controllers |
| `MonolithicODESolver` | Single `solve_ivp` co-integrating CV species, inter-CV transport, and controller differential state | Tight feedback loops (T_c < dt_h), multi-rate digital controllers, co-integrated integral states (PI/PID, observers) |

All five call `cv._advance_unchecked(...)` internally — except
`MonolithicODESolver`, which integrates every CV directly via
`cv.compute_rhs()` inside its shared `solve_ivp` call and therefore
**raises** if a per-CV `Simulation(solver=...)` is also configured
(there's no legitimate way for it to be silently correct, since it
never consults that config).

## Choosing a solver

| If your model has… | Start with |
|---|---|
| Mild kinetics, small `dt_h`, no special needs | `cv.advance()` (`SequentialAdvanceSolver`, the default) |
| Aggressive feeds or strong feed/reaction coupling, larger `dt_h` | `SimultaneousEulerSolver` |
| Stiff fast kinetics (BSM2 H₂), pH-coupled transfer, long timesteps | `SimultaneousAdaptiveSolver(method="Radau")` or `method="BDF"` |
| Reproducing BSM2 / PyADM1 validation results | `SimultaneousAdaptiveSolver(freeze_speciation=True, method="Radau")` |
| τ_CFL ≫ dt_h, slow controllers | `ExplicitEulerSystemSolver` (the default) |
| Fast inter-CV circulation | `MultirateSystemSolver` |
| Tight feedback loops / multi-rate digital controllers | `MonolithicODESolver` |
| Reproducing a reference model's own discrete interleaving convention | Write your own `SystemSolver` — see [01_writing_a_custom_solver.ipynb](01_writing_a_custom_solver.ipynb) |

If a model works with the defaults and the results look right, there's
no reason to switch. The non-default solvers exist because real models
exist that the defaults cannot integrate accurately or stably.

## Cross-references

- [`../../../docs/solvers.md`](../../../docs/solvers.md) — full
  solver landscape (strengths/weaknesses, configuration, accuracy +
  conservation monitoring) this notebook is a condensed map of.
- [`../../../docs/design/SOLVER_ARCHITECTURE.md`](../../../docs/design/SOLVER_ARCHITECTURE.md) —
  full two-axis design rationale, the DAE/SUNDIALS Phase F/G
  placeholders, and identified-but-unbuilt extensions (SIA, reactive
  D_eff).
- [`../../../docs/phases-shipped/STEP_SOLVER_INTERFACE_REFINEMENT.md`](../../../docs/phases-shipped/STEP_SOLVER_INTERFACE_REFINEMENT.md) —
  the design review that shipped the ownership guard, the
  gas/liquid generalization, the shared `clamp_fn` module, and this
  folder's walkthrough notebook.
- [`../../../docs/phases-shipped/ORDERING.md`](../../../docs/phases-shipped/ORDERING.md) —
  `SequentialAdvanceSolver`'s operator-splitting ordering rationale,
  inverted by [01_writing_a_custom_solver.ipynb](01_writing_a_custom_solver.ipynb)'s
  Axis 1 example.